# AutoLyap: Iteration-independent Lyapunov analyses - Example 1

[Open in Colab](https://colab.research.google.com/github/ManuUpadhyaya/tutorial_ELLIIT_2026/blob/main/notebooks/iteration_independent_lyapunov_example_1.ipynb)

This notebook reproduces the Davis--Yin three-operator splitting example from the iteration-independent Lyapunov analysis slides. It is fully filled in: there are no exercise placeholders.


## Setup

Install AutoLyap. The notebook uses `backend="cvxpy"` and `cvxpy_solver="CLARABEL"`, which is suitable for Google Colab and does not require a commercial SDP solver.


In [ ]:
%pip install -q autolyap


## Problem setup

Consider the three-function composite minimization problem

$$
\underset{x\in\mathcal{H}}{\operatorname{minimize}}\quad
f_1(x)+f_2(x)+f_3(x),
$$

where

$$
 f_1\in\mathcal{F}_{0,L_1}(\mathcal{H}),\qquad
 f_2\in\mathcal{F}_{\mu_2,L_2}(\mathcal{H}),\qquad
 f_3\in\mathcal{F}_{0,\infty}(\mathcal{H}),
$$

with $L_1>0$ and $0<\mu_2\leq L_2<+\infty$. Davis--Yin three-operator splitting is

$$
\begin{aligned}
v^k &= \operatorname{prox}_{\gamma f_1}(x^k),\\
w^k &= \operatorname{prox}_{\gamma f_3}\left(2v^k-x^k-\gamma\nabla f_2(v^k)\right),\\
x^{k+1} &= x^k+\lambda(w^k-v^k),
\end{aligned}
$$

with $\gamma\in\mathbb{R}_{++}$ and $\lambda\in\mathbb{R}$. We use AutoLyap to search for the smallest feasible $\rho\in[0,1)$ such that

$$
\|v^k-y^\star\|^2\in\mathcal{O}(\rho^k)\quad\text{as}\quad k\to\infty,
$$

where $y^\star\in\operatorname{Argmin}_{y\in\mathcal{H}} f_1(y)+f_2(y)+f_3(y)$.


## State-space representation

$$
\begin{aligned}
\mathbf{x}^{k+1}
&=\left(\begin{bmatrix}1\end{bmatrix}\otimes I\right)\mathbf{x}^k
+\left(\begin{bmatrix}-\gamma\lambda&-\gamma\lambda&-\gamma\lambda\end{bmatrix}\otimes I\right)\mathbf{u}^k,\\
\mathbf{y}^{k}
&=\left(\begin{bmatrix}1\\1\\1\end{bmatrix}\otimes I\right)\mathbf{x}^k
+\left(\begin{bmatrix}
-\gamma&0&0\\
-\gamma&0&0\\
-2\gamma&-\gamma&-\gamma
\end{bmatrix}\otimes I\right)\mathbf{u}^k,\\
\mathbf{u}^k
&\in \partial f_1(y_1^k)\times\partial f_2(y_2^k)\times\partial f_3(y_3^k).
\end{aligned}
$$

where

$$
\begin{aligned}
\mathbf{x}^k &= x^k,\\
\mathbf{u}^k &= (u_1^k,u_2^k,u_3^k)\\
&=\left(
\frac{x^k-v^k}{\gamma},
\nabla f_2(v^k),
\frac{2v^k-x^k-\gamma\nabla f_2(v^k)-w^k}{\gamma}
\right),\\
\mathbf{y}^k &= (y_1^k,y_2^k,y_3^k)=(v^k,v^k,w^k).
\end{aligned}
$$

The built-in `DavisYin` class in AutoLyap implements exactly these matrices.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from autolyap import IterationIndependent as II
from autolyap import SolverOptions
import autolyap.algorithms as alg
import autolyap.problemclass as pc

# CLARABEL can return optimal_inaccurate on this SDP while still producing a
# feasible certificate. We record the solve status below instead of printing
# the same warning repeatedly during the sweep.
warnings.filterwarnings(
    "ignore",
    message="Solution may be inaccurate.*",
)


## Define the problem data

The slide sweep uses $\mu_2=1$, $L_2=2$, $\gamma=1/L_2$, and $\lambda=1$. We start with one representative value $L_1=1$.


In [ ]:
mu2 = 1.0
L2 = 2.0
L1 = 1.0
gamma = 1.0 / L2
lambda_value = 1.0

solver_options = SolverOptions(backend="cvxpy", cvxpy_solver="CLARABEL")


def make_problem(L1_value: float):
    return pc.InclusionProblem([
        pc.SmoothConvex(L=L1_value),
        pc.SmoothStronglyConvex(mu=mu2, L=L2),
        pc.Convex(),
    ])


algorithm = alg.DavisYin(gamma=gamma, lambda_value=lambda_value)
P, p, T, t = II.LinearConvergence.get_parameters_distance_to_solution(algorithm)


## Lower bound for linear convergence

For this example, AutoLyap chooses $P,p,T,t$ so that the lower bound in **C2** is the squared distance of the first output point to a solution:

$$
\left\langle \boldsymbol{\xi}^{k}_{\mathrm{vec}},
(P\otimes I)\boldsymbol{\xi}^{k}_{\mathrm{vec}}\right\rangle
+p^{\top}\boldsymbol{\xi}^{k}_{\mathrm{func}}
=\|y_1^k-y^\star\|^2=\|v^k-y^\star\|^2,
$$

and the residual lower bound in **C3** is zero:

$$
\left\langle \boldsymbol{\xi}^{k}_{\mathrm{vec}},
(T\otimes I)\boldsymbol{\xi}^{k}_{\mathrm{vec}}\right\rangle
+t^{\top}\boldsymbol{\xi}^{k}_{\mathrm{func}}=0.
$$

The bisection search then finds the smallest feasible $\rho$ for the Lyapunov inequality

$$
\mathcal{V}(k+1)\leq \rho\mathcal{V}(k)-\mathcal{R}(k).
$$


In [ ]:
def solve_rate_for_L1(L1_value: float, tol: float = 2e-3):
    problem = make_problem(float(L1_value))
    result = II.LinearConvergence.bisection_search_rho(
        problem,
        algorithm,
        P,
        T,
        p=p,
        t=t,
        S_equals_T=True,
        s_equals_t=True,
        remove_C3=True,
        lower_bound=0.0,
        upper_bound=1.0,
        tol=tol,
        solver_options=solver_options,
        verbosity=0,
    )
    if result["status"] != "feasible":
        raise RuntimeError(
            f"No feasible Lyapunov certificate for L1={L1_value:.6g}."
        )
    return result


single_result = solve_rate_for_L1(L1)
print(f"status: {single_result['status']}")
print(f"solve_status: {single_result['solve_status']}")
print(f"rho (AutoLyap, L1={L1:g}): {float(single_result['rho']):.6f}")


## Comparison bounds

The slide compares the AutoLyap certificate with the two standard bounds

$$
\rho_{\mathrm{DY}}
=1-\frac{\mu_2}{L_2(1+L_1/L_2)^2},
\qquad
\rho_{\mathrm{PG}}
=1-\min\left\{\frac{\mu_2}{L_2},\frac{1}{1+L_1/L_2}\right\}.
$$


In [ ]:
def rho_dy_bound(L1_value):
    L1_value = np.asarray(L1_value, dtype=float)
    return 1.0 - mu2 / (L2 * (1.0 + L1_value / L2) ** 2)


def rho_pg_bound(L1_value):
    L1_value = np.asarray(L1_value, dtype=float)
    return 1.0 - np.minimum(mu2 / L2, 1.0 / (1.0 + L1_value / L2))


print(f"rho_DY (L1={L1:g}): {float(rho_dy_bound(L1)):.6f}")
print(f"rho_PG (L1={L1:g}): {float(rho_pg_bound(L1)):.6f}")


## Sweep over $L_1$

The slide considers $L_1\in(0,40]$. The default grid starts at $0.1$ rather than exactly at zero because the SDP is numerically more delicate extremely close to $L_1=0$. Increase `num_L1_values` for a smoother plot.


In [ ]:
num_L1_values = 100
L1_values = np.linspace(0.1, 40.0, num_L1_values)

rho_autolyap_values = []
solve_statuses = []
for idx, L1_value in enumerate(L1_values, start=1):
    result = solve_rate_for_L1(float(L1_value), tol=2e-3)
    rho_autolyap_values.append(float(result["rho"]))
    solve_statuses.append(result["solve_status"])

    if idx == 1 or idx % 5 == 0 or idx == len(L1_values):
        print(f"Solved {idx:>2}/{len(L1_values)}")

rho_autolyap_values = np.array(rho_autolyap_values)
status_counts = {status: solve_statuses.count(status) for status in sorted(set(solve_statuses))}
print("solve_status counts:", status_counts)


In [ ]:
L1_curve = np.linspace(0.1, 40.0, 400)

fig, ax = plt.subplots(figsize=(7, 3.8), dpi=160, constrained_layout=True)
ax.plot(
    L1_curve,
    rho_dy_bound(L1_curve),
    linewidth=2.0,
    color="#6f5aa7",
    label=r"$\rho_{\mathrm{DY}}$",
)
ax.plot(
    L1_curve,
    rho_pg_bound(L1_curve),
    linewidth=2.0,
    color="#c8a437",
    label=r"$\rho_{\mathrm{PG}}$",
)
ax.scatter(
    L1_values,
    rho_autolyap_values,
    s=24,
    color="#4f8bc9",
    label="AutoLyap",
    zorder=3,
)
ax.set_xlabel(r"$L_1$")
ax.set_ylabel(r"$\rho$", rotation=0)
ax.set_title(r"Davis--Yin splitting: certified linear rate")
ax.set_xlim(0.0, 40.0)
ax.set_ylim(0.0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right")
plt.show()
